# Prepare files for different analysis

In [1]:
import reframed
from reframed import SteadyCom
import pandas as pd
import numpy as np
from os import listdir
import warnings
import seaborn as sns
from collections import OrderedDict
import json

##  Prepare files for SMETANA 

We need:
- Community models
- Community data file

We also need:
- Media

#### Load data and models

**Load relative abundance (E. coli)**

In [2]:
relative_abundance_ecoli = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance_ecoli.index = relative_abundance_ecoli.index.map(lambda x:x if x!="yhhK" else "panZ")

**Load models**

*E. coli mutants  (top 7)*

In [3]:
RA_ecoli_mean = relative_abundance_ecoli.T.mean().sort_values(ascending=False)
top7 = RA_ecoli_mean[RA_ecoli_mean>0.01].index
top7_string = [f"mutant_{mutant}.xml" for mutant in top7]

In [4]:
models_ecoli = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Guillen2021/strains/"):
        if path not in top7_string: continue
        models_ecoli[path.split("_")[1][:-4]] = reframed.load_cbmodel("../models/Guillen2021/strains/"+path,
                                                        exchange_detection="^R_EX_(?!nad_c$|thf_c$).*")

*Gut microbiome members*

In [5]:
models_gut = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Venturelli2018/species_used_noO2/"):
        models_gut[path[:-4]] = reframed.load_cbmodel("../models/Venturelli2018/species_used_noO2/"+path,
                                                      exchange_detection="R_EX_(?!biomass__40__e__41__)")


**Load media**

*For E. coli mutants (M9 medium to force growth of specific members)*

In [6]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="M9"].compound.values)
env_ecoli = reframed.Environment.from_compounds(compounds,max_uptake=10)

*For gut microbiome members*

In [7]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
#reactions = [rxn.replace("40","91").replace("41","93") for rxn in reactions]

env_gut = reframed.Environment.from_reactions(reactions,max_uptake=10)

**Load other data**

In [8]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

#### E.coli mutants - Prepare files

In [ ]:
model_ids = [model.id for model in models_ecoli.values()]
community_ids = ["ecoli_auxotrophs" for model in models_ecoli.values()]

pd.DataFrame([community_ids,model_ids]).T.to_csv("../data/Guillen2021/generated/communities_smetana.tsv",
                                        index=None,
                                        header=None,
                                        sep="\t")

#### Gut microbiome - Prepare files

In [ ]:
df_comms = []

# Full community
model_ids = [model_id for model_id in models_gut.keys()]
community_ids = ["Full" for model_id in model_ids]
df_comm = pd.DataFrame([community_ids,model_ids])
df_comms.append(df_comm)

# Reduced communities
for model_id in models_gut.keys():
    model_ids = list(models_gut.keys())
    model_ids.remove(model_id)
    community_id = model_id.split("_")[0][0].upper() + model_id.split("_")[1][0].upper()+"-"
    community_ids = [community_id for id_ in model_ids]
    
    df_comm = pd.DataFrame([community_ids,model_ids])
    df_comms.append(df_comm)

# Save results
communities_gut = pd.concat(df_comms,axis=1).T
print(communities_gut[0].unique())
communities_gut.to_csv("../data/Venturelli2018/generated/communities_smetana.tsv",
                                        index=None,
                                        header=None,
                                        sep="\t")



In [ ]:
# Media data
mets_gut = []
for rxn in env_gut.keys():
    mets_gut.append(rxn.replace("R_EX_","").replace("__91__e__93__",""))

media_name = ["LB_extend" for met in mets_gut]
media_db_gut_smetana = pd.DataFrame({"medium":media_name,"compound":mets_gut})
media_db_gut_smetana.to_csv("../data/Venturelli2018/generated/media_db_smetana.tsv",
                            index=False,
                            sep="\t")

## Prepare files for steadiercom

We need:
* Media_db (already have)
* Communities-file with relative abundance

In [22]:
with open("../data/inorganic.txt") as f:
    mets = f.read().split("\n")

mets_gut = [f"R_EX_{met}__91__e__93__" for met in mets]

with open("../data/Venturelli2018/generated/unlimited.txt",'w') as f:
    text = "\n".join(mets_gut)
    f.write(text)



In [24]:
mets_gut = [f"R_EX_{met}_e" for met in mets]

with open("../../SteadierCom/tests/data/unlimited_R_prefix.txt",'w') as f:
    text = "\n".join(mets_gut)
    f.write(text)


#### E.coli mutants

In [2]:
# NB! yhhK in data == panZ in paper
relative_abundance = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance.index =relative_abundance.index.map(lambda x:x if x!="yhhK" else "panZ")

In [3]:
relative_abundance_ecoli = relative_abundance.T.mean().sort_values(ascending=False).head(7)
relative_abundance_ecoli = relative_abundance_ecoli.reset_index()
relative_abundance_ecoli["community"] = relative_abundance_ecoli["index"].map(lambda x:"Ecoli_mutants")
relative_abundance_ecoli["index"] = relative_abundance_ecoli["index"].map(lambda x:"mutant_"+x)
relative_abundance_ecoli\
        .set_index("community")\
        .reset_index()\
        .to_csv("../data/Guillen2021/generated/communities_steadiercom.tsv",
                                                     sep="\t",
                                                     header=None,
                                                     index=None)


#### Gut microbiome

**Processing datasetev3**

In [4]:
minimal_medium = pd.read_csv("../output/Venturelli2018/minimal_media.tsv",sep="\t",index_col=0)
member_id2member_name = {x.split("_")[0][0]+x.split("_")[1][0].upper():x for x in minimal_medium.columns}


datasetev3 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0004-datasetev3.xlsx",sheet_name="Dataset EV3")

abundance_dict = {}
last_item = "'ER-'"
abundance_dict[last_item]=[]
for i,row in datasetev3.iterrows():
    list_ = row.iloc[0].split("    ")
    
    if len(list_)==1:
        last_item =list_[0]
        abundance_dict[last_item]=[]
    else:
        abundance_dict[last_item].append(list_[1:])

In [5]:
abundance_dataframes = {community:pd.DataFrame(abundances).map(lambda x: float(x)) for community,abundances in abundance_dict.items()}

for community,dataframe in abundance_dataframes.items(): dataframe.columns=["BH","CA","BU","PC","BO","BV","BT","EL","FP","CH","DP","ER"] 

for community,dataframe in abundance_dataframes.items(): dataframe.columns = dataframe.columns.map(lambda x:member_id2member_name[x])

abundance_series = {community_id.replace("'",""):dataframe.iloc[-1,:] for community_id,dataframe in abundance_dataframes.items()}
for community,series in abundance_series.items(): 
    if community!="NONE": 
        abundance_series[community]=series.drop(member_id2member_name[community[:-1]])


communities_exp_abundance = pd.concat(abundance_series)
communities_exp_abundance

ER-   Blautia_hydrogenotrophica_DSM_10507       0.0109
      Collinsella_aerofaciens_ATCC_25986        0.0255
      Bacteroides_uniformis_ATCC_8492           0.3809
      Prevotella_copri_CB7_DSM_18205            0.0003
      Bacteroides_ovatus_ATCC_8483              0.1343
                                                 ...  
NONE  Eggerthella_lenta_DSM_2243                0.0168
      Faecalibacterium_prausnitzii_A2_165       0.2393
      Clostridium_hiranonis_TO_931_DSM_13275    0.0212
      Desulfovibrio_piger_ATCC_29098            0.0019
      Eubacterium_rectale_ATCC_33656            0.0086
Name: 6, Length: 144, dtype: float64

In [11]:
for community_id in communities_exp_abundance.index.get_level_values(0).unique():
    
    new_id = community_id.replace("-","_minus")
    communities_exp_abundance[[community_id]].to_csv(f"../data/Venturelli2018/generated/communities_steadiercom/{new_id}_abundance.tsv",
                                                     sep="\t",
                                                     header=None)
    
    communities_exp_abundance[[community_id]].reset_index().iloc[:,:-1].to_csv(f"../data/Venturelli2018/generated/communities_steadiercom_no_abundance/{new_id}_abundance.tsv",
                                                     sep="\t",
                                                     header=None,
                                                     index=None)